# Session 15 lab: Classification, yes or no questions

You will fit a **logistic regression** and a **decision tree** on the Nile Microfinance loans, read the **predicted probability** each gives, and choose a **decision threshold** from the cost of a missed default and the cost of a false alarm. All data is fictional and the costs are illustrative teaching numbers.

Rhythm for every step: **PREDICT** (write your guess), **run**, **READ** the output, **CHANGE** one thing. Cells with `# your code here` hold a placeholder: replace that line with your answer and run the cells in order. The `check(...)` lines print NOT YET until your number is right. What you keep: this notebook, `loan_risk_scores.csv` and a three-line threshold note.

One rule for the whole lab: the **test loans are used once**, at the end. The threshold is chosen on the training loans.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
import numpy as np
import pandas as pd

# Run this cell once. It lets a check say NOT YET instead of stopping the notebook when your answer is still empty.
def safe(f):
    try:
        return f()
    except Exception:
        return np.nan

---
## Step 1: Prepare the loans and one-hot encode the text

The same preparation as Session 14, so that every session works on the same 8,000 loans. A model needs clean numbers.

**PREDICT first.** The file has 8,000 loans. Two have an impossible age (0 and 199). If we replace those ages instead of deleting the loans, how many loans remain?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
loans = pd.read_csv("loans.csv")
print(loans.shape)
bad_age = (loans.age < 18) | (loans.age > 100)                      # two impossible ages: 0 and 199
loans.loc[bad_age, "age"] = loans.loc[~bad_age, "age"].median()      # the median of the valid ages (36); all 8,000 loans stay
loans["income_missing"] = loans.monthly_income_ugx.isna().astype(int)   # flag first, then fill
median_income = np.nan   # your code here: the median of monthly_income_ugx
loans["monthly_income_ugx"] = loans.monthly_income_ugx.fillna(median_income)
# the instalment-to-income ratio. The 1.15 is an illustrative assumption for interest and fees, not in the data.
loans["instalment_to_income"] = np.nan   # your code here: loan_amount_ugx * 1.15 / term_months / monthly_income_ugx
print(len(loans), "loans;", loans.income_missing.sum(), "incomes were blank")
print(loans["instalment_to_income"].describe().round(2))

In [ ]:
check("loans after the preparation", len(loans), 8000, 0)
check("default rate", loans["default"].mean(), 0.1255, 0.001)
check("median income (UGX)", safe(lambda: median_income), 642600.0, 1)
check("median instalment-to-income ratio", safe(lambda: loans["instalment_to_income"].median()), 0.29162113874253703, 0.005)

**READ the output.** The default rate is about 12.6 percent. Which class is the positive class, and why does the rarity of default matter for a model?

> My answer: 

Next, **one-hot encoding**. We keep the 12 numeric columns of Session 14 and turn the three text variables (gender, occupation, loan_purpose) into 0 and 1 columns. The district (30 values) stays out, and so does `collection_calls`: it exists only after a loan goes bad, and Session 16 explains why it must stay out.

In [ ]:
FEATURES = ["age", "monthly_income_ugx", "income_missing", "loan_amount_ugx", "term_months", "previous_loans",
            "previous_late_payments", "group_loan", "savings_balance_ugx", "has_mobile_money", "distance_to_branch_km", "instalment_to_income"]
X = loans[FEATURES]   # your code here: join the 12 FEATURES to pd.get_dummies(loans[["gender", "occupation", "loan_purpose"]], dtype=int) with pd.concat(..., axis=1)
y = loans["default"]
print(X.shape)

In [ ]:
check("columns of X after one-hot encoding", X.shape[1], 27, 0)
check("a Teacher column exists", int("occupation_Teacher" in X.columns), 1, 0)

**CHANGE one thing.** Print `X.columns` and find the columns made from `loan_purpose`. How many are there, and why is there no column called `loan_purpose`?

> What I saw: 

---
## Step 2: Split the loans and fit a logistic regression

Hide a quarter of the loans, learn from the rest, then ask the model for a probability of default for each hidden loan. The model is a pipeline: StandardScaler puts every variable on the same scale, then LogisticRegression runs with its defaults (L2 penalty, C = 1.0: regularisation). That is why the coefficients are per 1 standard deviation.

**PREDICT first.** Will the coefficient of instalment_to_income be positive or negative? A positive coefficient raises the log-odds of default.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = X.iloc[:2000], X.iloc[2000:4000], y.iloc[:2000], y.iloc[2000:4000]   # your code here: train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(len(X_train), len(X_test), int(y_test.sum()))
print("first test rows:", X_test.index[:10].tolist())   # with stratify=y and random_state=42 these are the rows of the course notes

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=0.000001))   # your code here: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)). The placeholder has a very strong penalty (C=0.000001)
model.fit(X_train, y_train)
y_prob = np.full(len(X_test), 0.5)   # your code here: column 1 of model.predict_proba(X_test) is the probability of default
coef = pd.Series(model[-1].coef_[0], index=X.columns).sort_values()   # per 1 standard deviation of each variable
print(coef.round(2))
auc = safe(lambda: roc_auc_score(y_test, y_prob))
print("average predicted probability:", round(float(y_prob.mean()), 3), " test AUC:", auc)

In [ ]:
check("loans in X_train", len(X_train), 6000, 0)
check("defaults in y_test", int(y_test.sum()), 251, 0)
check("average predicted probability", safe(lambda: y_prob.mean()), 0.12660815729440855, 0.01)
check("coefficient of instalment_to_income", safe(lambda: coef["instalment_to_income"]), 0.8012903288258012, 0.05)
check("test AUC", auc, 0.7557944323335588, 0.01)

**READ the output.** Which three variables push the log-odds of default up most, and which push it down most? Does that match the bar charts in the slides? Why are the two gender coefficients about 0?

> My answer: 

**CHANGE one thing.** Compute the odds ratio of the ratio with `np.exp(coef["instalment_to_income"])`. In words: how many times larger are the odds of default for each extra standard deviation of the ratio?

> What I saw: 

---
## Step 3: Fit a decision tree and read its rules

A tree asks yes or no questions. You can read every path.

**PREDICT first.** Which variable will the root node use for the first split?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=1, random_state=42)   # your code here: DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=list(X.columns)))
tree_prob = np.full(len(X_test), 0.5)   # your code here: the probability of default, column 1 of tree.predict_proba(X_test)
print("different probabilities the tree gives:", len(set(tree_prob.round(6))))

In [ ]:
check("tree depth", tree.get_depth(), 3, 0)
check("leaf nodes", tree.get_n_leaves(), 8, 0)
check("different probabilities", len(set(tree_prob.round(6))), 8, 0)
check("average tree probability", safe(lambda: tree_prob.mean()), 0.1244943229309741, 0.01)

**READ the output.** Pick one path from the root to a leaf and say it as a sentence: "If the instalment-to-income ratio is ..., and ..., the model says ... percent of such loans default."

> My answer: 

**CHANGE one thing.** Refit with max_depth=2. Store the number of leaf nodes in `leaves_depth2`.

> What I saw: 

In [ ]:
tree2 = DecisionTreeClassifier(max_depth=2, random_state=42)
tree2.fit(X_train, y_train)
leaves_depth2 = np.nan   # your code here: the number of leaf nodes of tree2 (tree2.get_n_leaves())
print(leaves_depth2)
check("leaves at depth 2", safe(lambda: leaves_depth2), 4, 0)

---
## Step 4: Choose the decision threshold from the cost, on the training loans

A missed default costs the lender far more than a false alarm. Put illustrative costs on both. Choose the threshold WITHOUT the test loans: use cross-validated probabilities on the training loans, and compare with the cost rule. Then report the cost on the test loans once.

**PREDICT first.** Count the test loans flagged at a decision threshold of 0.20. Will the model flag more or fewer than 100 loans? How many of the 251 defaults will it catch?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
flag = y_prob >= 0.20
actual = (y_test == 1).to_numpy()
tp = np.nan   # your code here: loans flagged AND really defaulted (flag & actual)
fp = np.nan   # your code here: loans flagged but repaid (flag & ~actual)
fn = np.nan   # your code here: loans not flagged but defaulted (~flag & actual)
print("flagged:", int(flag.sum()), " caught:", tp, " false alarms:", fp, " missed:", fn)

In [ ]:
check("defaults caught at 0.20", safe(lambda: tp), 108, 4)
check("false alarms at 0.20", safe(lambda: fp), 170, 8)
check("defaults missed at 0.20", safe(lambda: fn), 143, 4)

This table only describes what happens on the test loans. It does not choose the threshold. Now price the mistakes. These costs are **illustrative**: a missed default loses UGX 1,000,000 of principal and a false alarm loses UGX 100,000 of interest.

Step A: get an honest probability for every TRAINING loan with **cross-validation** (`cross_val_predict`: five rounds, each loan scored by a model that did not see it).

In [ ]:
from sklearn.model_selection import cross_val_predict

p_cv = np.full(len(y_train), 0.5)   # your code here: cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1]
actual_train = (y_train == 1).to_numpy()
print("training loans:", len(p_cv), " average out-of-fold probability:", round(float(p_cv.mean()), 3))

In [ ]:
check("out-of-fold probabilities for every training loan", len(p_cv), 6000, 0)
check("average out-of-fold probability", safe(lambda: p_cv.mean()), 0.1255270608251337, 0.01)

Step B: the cost of the mistakes at every threshold on the training loans (scaled to 2,000 loans so that it compares with the test loans).

In [ ]:
def total_cost(prob, actual, threshold, cost_miss, cost_alarm):
    flagged = prob >= threshold
    missed = 0   # your code here: defaults not flagged: int((~flagged & actual).sum())
    alarms = 0   # your code here: repaid loans flagged: int((flagged & ~actual).sum())
    return missed * cost_miss + alarms * cost_alarm

thresholds = np.round(np.arange(0.02, 0.605, 0.01), 2)
scale = len(y_test) / len(y_train)
costs_cv = [total_cost(p_cv, actual_train, t, 1_000_000, 100_000) * scale for t in thresholds]
cv_threshold = np.nan   # your code here: the threshold with the smallest cost (np.argmin)
rule = 100_000 / (100_000 + 1_000_000)
print("cheapest threshold on the training loans:", cv_threshold, " cost per 2,000 loans:", round(min(costs_cv) / 1e6, 1), "million")
print("cost rule: 100,000 / (100,000 + 1,000,000) =", round(rule, 4))

In [ ]:
check("threshold from cross-validation", safe(lambda: cv_threshold), 0.1, 0.03)
check("training cost at that threshold (UGX million per 2,000 loans)", safe(lambda: min(costs_cv) / 1e6), 135.56666666666666, 4)
check("cost rule", rule, 0.09090909090909091, 0.001)

**READ the output.** Run the next cell. Where is the curve lowest? How far is that from the default threshold of 0.50, and from the cost rule?

> My answer: 

In [ ]:
import matplotlib.pyplot as plt

plt.plot(thresholds, np.array(costs_cv) / 1e6)
plt.axvline(rule, linestyle=":", label="cost rule")
plt.axvline(0 if np.isnan(cv_threshold) else cv_threshold, linestyle="--", label="cross-validation")
plt.xlabel("decision threshold")
plt.ylabel("cost per 2,000 loans, UGX million")
plt.title("Cost of mistakes on the training loans (illustrative costs)")
plt.legend()
plt.show()

Step C: only now use the test loans, **once**, to report the cost of the choices you made.

In [ ]:
cost_default = total_cost(y_prob, actual, 0.50, 1_000_000, 100_000) / 1e6
cost_cv_choice = total_cost(y_prob, actual, cv_threshold, 1_000_000, 100_000) / 1e6
cost_rule_choice = total_cost(y_prob, actual, rule, 1_000_000, 100_000) / 1e6
print("test cost, UGX million. At 0.50:", round(cost_default, 1), " at the cross-validated threshold:", round(cost_cv_choice, 1), " at the rule:", round(cost_rule_choice, 1))

In [ ]:
check("test cost at 0.50 (UGX million)", safe(lambda: cost_default), 201.6, 6)
check("test cost at the cross-validated threshold (UGX million)", safe(lambda: cost_cv_choice), 137.9, 6)
check("test cost at the cost rule (UGX million)", safe(lambda: cost_rule_choice), 133.9, 6)

**CHANGE one thing.** Make a false alarm cost 500,000 instead of 100,000 and store the new cheapest threshold on the training loans in `cv_threshold_500k`. Predict first: higher or lower than before? What does the rule give?

> What I saw: 

In [ ]:
costs_500k = [total_cost(p_cv, actual_train, t, 1_000_000, 500_000) * scale for t in thresholds]
cv_threshold_500k = np.nan   # your code here: the threshold with the smallest cost in costs_500k
print(cv_threshold_500k, " rule:", round(500_000 / 1_500_000, 3))
check("cheapest threshold when a false alarm costs 500,000", safe(lambda: cv_threshold_500k), 0.31, 0.05)

---
## Step 5: Keep: a scored file and a threshold note

This is what you can hand to a colleague in credit.

In [ ]:
chosen = round(cv_threshold, 2)
scores = pd.DataFrame({"loan_id": [], "probability": [], "flag": []})   # your code here: a DataFrame with columns loan_id, probability and flag (1 when probability >= chosen). Use loans.loc[X_test.index, "loan_id"]
scores.to_csv("loan_risk_scores.csv", index=False)
print(scores.head())
print("loans flagged:", int(scores["flag"].sum()), "of", len(scores))
import os
check("rows in loan_risk_scores.csv", len(scores), 2000, 0)
check("file written", int(os.path.exists("loan_risk_scores.csv")), 1, 0)
check("loans flagged at the chosen threshold", int(scores["flag"].sum()), 794, 60)

In [ ]:
show_checks()

### My threshold note

Write three lines.

1. **Threshold:** flag a loan when its predicted probability is at least ... (chosen on the training loans by ...)
2. **Cost:** on the 2,000 test loans this costs about UGX ... million, against UGX ... million at 0.50.
3. **Caution:** ...

**Download** `loan_risk_scores.csv` from the Colab file panel and keep it with this notebook. Next session: how do we tell a model that works from one that only looks good?